<a href="https://colab.research.google.com/github/RasanaPJ/Deep-Learning/blob/master/mFinAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mFinAgent — Google ADK + Claude on Colab

A conversational AI agent for **return** and **risk** analysis of NSE stocks.

What you'll see by the end:
- 3 case-study tools wired into a Google ADK `LlmAgent`
- A SQLite-backed `DatabaseSessionService` so sessions survive restarts
- A multi-turn demo proving the agent reuses prior tool results from the session transcript

> *Educational. Not investment advice. Past returns do not predict future performance.*


## 1. Install dependencies

In [1]:
# Run once per Colab runtime. ~30s.
%pip install -q 'google-adk[extensions]' yfinance pydantic nest_asyncio aiosqlite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 938.0/938.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.9/154.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/

## 2. Configure GROQ API key


In [2]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('GROQ_API_KEY: ')

GROQ_API_KEY: ··········


In [3]:
# Optional: override the LiteLLM model string
os.environ.setdefault('MFINAGENT_MODEL', 'groq/llama-3.3-70b-versatile')
print('OK — model:', os.environ['MFINAGENT_MODEL'])

OK — model: groq/llama-3.3-70b-versatile


## 3. Imports + nested asyncio (Colab needs this)

In [4]:
import asyncio, json, math, uuid
from datetime import datetime, timedelta
from difflib import SequenceMatcher
from typing import List, Optional

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import DatabaseSessionService
from google.genai import types
import difflib
import yfinance as yf
import pandas as pd


import nest_asyncio
nest_asyncio.apply()  # lets us 'await' inside notebook cells alongside Colab's loop

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## 4. The Two case-study tools

Each one is a **plain Python function**. ADK introspects type hints +
docstring to build the JSON schema the model sees, so the docstrings are
written *for the model* (when to use, not just what it does).

In [6]:
# ── Top NSE companies with their common aliases ──
# In production this would be ~50 names; trimmed here for clarity.
NSE_TICKERS = {
    "RELIANCE.NS":   ["Reliance", "Reliance Industries", "RIL"],
    "HDFCBANK.NS":   ["HDFC Bank", "HDFC Bank Limited"],
    "TCS.NS":        ["TCS", "Tata Consultancy Services"],
    "INFY.NS":       ["Infosys", "Infy"],
    "ICICIBANK.NS":  ["ICICI Bank", "ICICI"],
    "SBIN.NS":       ["State Bank of India", "SBI"],
    "TATAMOTORS.NS": ["Tata Motors"],
    "TATASTEEL.NS":  ["Tata Steel"],
    "BAJFINANCE.NS": ["Bajaj Finance"],
    "LT.NS":         ["Larsen & Toubro", "L&T"],
}


# Resolves a free-text company name to an NSE ticker using exact, substring,
# regex on each alias and fuzzy fallback
# Returns the best possible single ticker, with highest score

def lookup_ticker(query: str) -> dict:
    """Resolve a free-text Indian company name to NSE tickers.

    USE FIRST when the user names a stock — before fetching prices.
    Tolerates typos and partial names. Returns ranked candidate.

    Args:
        query: Free-text company name (e.g. 'hdfc', 'Relaince', 'tata').
    """
    q = query.strip().lower()
    cand = {}

    # Stage 1+2: exact, substring, regex on each alias
    for ticker, aliases in NSE_TICKERS.items():
        for a in aliases:
            al = a.lower()
            score = 1.0 if al == q else (
                    0.85 if q in al else (
                    0.80 if al in q else 0.0))
            if score > cand.get(ticker, {}).get("score", 0):
                cand[ticker] = {"ticker": ticker, "name": a, "score": score}

    # Stage 3: fuzzy fallback if nothing matched
    if not cand:
        flat = [(t, a) for t, al in NSE_TICKERS.items() for a in al]
        names = [a for _, a in flat]
        for m in difflib.get_close_matches(query, names, n=top_k, cutoff=0.6):
            t = next(t for t, a in flat if a == m)
            score = difflib.SequenceMatcher(None, q, m.lower()).ratio()
            cand.setdefault(t, {"ticker": t, "name": m, "score": round(score, 2)})

    ranked = sorted(cand.values(), key=lambda x: -x["score"])[:1]
    return {"query": query, "matches": ranked}

# Pulls actual monthly close-to-close returns from yfinance and computes few
# basic stats and returns
# - monthly return
# - annualised version = monthly std × √12 — standard finance convention)
# - Min / max** month

def fetch_stock_stats(ticker: str, months: int) -> dict:
    """Fetch monthly close-to-close returns (in %) for an NSE ticker and
    calculates mean, annualised std (risk).

    Args:
        ticker: NSE symbol with .NS suffix (e.g. 'HDFCBANK.NS').
        months: Number of months of history to fetch.
    """
    end = pd.Timestamp.today()
    start = end - pd.DateOffset(months=months + 2)  # buffer for monthly resampling

    df = yf.download(ticker, start=start, end=end, interval="1mo",
                     progress=False, auto_adjust=True)
    if df.empty:
        # Errors are RETURNED (not raised) so the model can read and recover
        return {"error": f"No data for {ticker}"}

    # yfinance now returns a per-ticker MultiIndex / single-column DataFrame
    # for single-ticker calls; .squeeze() collapses it back to a Series so
    # iteration yields (Timestamp, value) pairs.
    closes = df["Close"].squeeze().dropna().tail(months + 1)
    rets = (closes.pct_change().dropna() * 100).round(2)

    n = len(rets)
    if n == 0:
        return {"error": "Empty returns list"}

    mean = sum(rets) / n
    var  = sum((r - mean) ** 2 for r in rets) / max(n - 1, 1)
    std  = var ** 0.5

    out = {
        "n_months": n,
        "mean_monthly_pct": round(mean, 2),
        "std_monthly_pct": round(std, 2),
        "annualised_std_pct": round(std * math.sqrt(12), 2),
        "min_pct": round(min(rets), 2),
        "max_pct": round(max(rets), 2),
    }

    return out

In [ ]:
# Quick smoke test (no network needed for the first two)
print(lookup_ticker('hdfc bank'))
print(fetch_stock_stats("HDFCBANK.NS", 6))

{'query': 'hdfc bank', 'matches': [{'ticker': 'HDFCBANK.NS', 'name': 'HDFC Bank', 'score': 1.0}]}
{'n_months': 6, 'mean_monthly_pct': -3.88, 'std_monthly_pct': 7.91, 'annualised_std_pct': 27.4, 'min_pct': -17.6, 'max_pct': 5.49}


## 5. Build the LlmAgent

Single `LlmAgent` with three tools. The agent replies in natural language —
no structured-output schema, no Pydantic validation step. The Runner replays
prior session events into the LLM context every turn, so the model already
sees its earlier tool results in the conversation transcript and can answer
follow-up questions without re-fetching.

In [7]:
_INSTRUCTION = '''\
You are mFinAgent, a conversational analyst for Indian equities (NSE)
who helps retail investors understand return and risk.

- For a new stock, default to a 12-month window unless the user asks otherwise.
- For comparative follow-ups ("vs the previous one", "in comparison"), reuse
  tool results already present in the conversation. Do NOT re-fetch prices
  for stocks already analysed earlier in this session.
- Be honest about uncertainty. Never fabricate numbers.
'''


# ADK injects each tool's schema (from its signature + docstring), so the
# instruction above only needs to cover behaviour the model cannot infer
# from those.
root_agent = LlmAgent(
    name='mFinAgent',
    model=LiteLlm(model=os.environ['MFINAGENT_MODEL']),
    description='Conversational return-and-risk analyst for NSE stocks.',
    instruction=_INSTRUCTION,
    tools=[
        lookup_ticker,
        fetch_stock_stats
    ],
)

In [8]:
print('agent ready:', root_agent.name, '| tools:',
      [getattr(t, 'name', getattr(t, '__name__', repr(t))) for t in root_agent.tools])

agent ready: mFinAgent | tools: ['lookup_ticker', 'fetch_stock_stats']


## 6. Runner + SQLite session service + helpers

`DatabaseSessionService` against a local SQLite file. The `+aiosqlite`
URL is required because the service is async.

Below we expose four small helpers that read more naturally than the raw
ADK API:

- `new_session()` → creates a new session, sets it as current, returns the id
- `list_sessions()` → returns every session this user has stored
- `resume_session(id)` → makes a prior session the current one
- `end_session()` → just clears the current id; data is already persisted in SQLite
- `ask(text)` → sends one user turn, prints tool trace + final response

Sessions are persisted in `alphascout.db`, so `end_session` doesn't delete
anything — it's purely bookkeeping. Re-running the notebook later, you can
call `resume_session(id)` with an id from `list_sessions()` and continue
the conversation.

In [9]:
APP_NAME = 'mFinAgent'
USER_ID  = 'manu'
DB_URL   = 'sqlite+aiosqlite:///mfinagent.db'

session_service = DatabaseSessionService(db_url=DB_URL)

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

# Module-level current-session pointer so ask() can stay short.
CURRENT_SESSION: Optional[str] = None


async def new_session() -> str:
    """Create a fresh session and make it the current one."""
    global CURRENT_SESSION
    sid = f'sess-{uuid.uuid4().hex[:8]}'
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=sid, state={},
    )
    CURRENT_SESSION = sid
    print(f'[new session: {sid}]')
    return sid


async def list_sessions():
    """Return every session stored for this user, most-recent first."""
    listing = await session_service.list_sessions(app_name=APP_NAME, user_id=USER_ID)
    sessions = list(listing.sessions or [])
    sessions.sort(key=lambda s: getattr(s, 'last_update_time', 0.0) or 0.0, reverse=True)
    for s in sessions:
        marker = '*' if s.id == CURRENT_SESSION else ' '
        print(f'  {marker} {s.id}  updated={getattr(s, "last_update_time", 0.0)}')
    return sessions


async def resume_session(session_id: str) -> str:
    """Make `session_id` the current session. Returns the id."""
    global CURRENT_SESSION
    sess = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id,
    )
    if sess is None:
        raise ValueError(f'session not found: {session_id}')
    CURRENT_SESSION = session_id
    n_events = len(sess.events or [])
    print(f'[resumed session: {session_id}]  ({n_events} prior events)')
    return session_id


def end_session() -> None:
    """Stop targeting the current session. Its data stays in SQLite."""
    global CURRENT_SESSION
    sid = CURRENT_SESSION
    CURRENT_SESSION = None
    print(f'[ended session: {sid}]  (still persisted in {DB_URL})')


print('session helpers ready  ·  DB:', DB_URL)


session helpers ready  ·  DB: sqlite+aiosqlite:///mfinagent.db


## 7. The `ask()` helper

Sends one user turn to the runner and prints:
- the tool trace (so you can see what the model called),
- the final assistant response.

In [11]:
def _short_args(args: dict) -> str:
    bits = []
    for k, v in args.items():
        s = repr(v)
        if len(s) > 60:
            s = s[:57] + '...'
        bits.append(f'{k}={s}')
    return ', '.join(bits)


async def ask(text: str):
    if CURRENT_SESSION is None:
        raise RuntimeError('No active session. Call new_session() or resume_session(id) first.')
    print(f'\nyou> {text}')
    msg = types.Content(role='user', parts=[types.Part(text=text)])
    full = ''
    async for ev in runner.run_async(user_id=USER_ID, session_id=CURRENT_SESSION, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                fc = getattr(p, 'function_call', None)
                if fc is not None:
                    print(f'  · tool: {fc.name}({_short_args(dict(fc.args) if fc.args else {})})')
        if ev.is_final_response() and ev.content and ev.content.parts:
            for p in ev.content.parts:
                t = getattr(p, 'text', None)
                if t:
                    full += t
    print(f'\nalphascout> {full or "(no text)"}')
    return full

print('ask() ready')


ask() ready


## 8. Demo — Scenario 1: brand-new conversation

`new_session()` → two `ask()` turns → `end_session()`.

Watch the second turn: the agent has the prior turn's tool results in its
context, so it should answer the comparison **without** re-fetching HDFC
Bank prices.

In [16]:
await new_session()
await ask('What is the return of HDFC Bank in the last 12 months?')

[new session: sess-1339cbe9]

you> What is the return of HDFC Bank in the last 12 months?
  · tool: lookup_ticker(query='HDFC Bank')
  · tool: fetch_stock_stats(months=12, ticker='HDFCBANK.NS')

alphascout> The return of HDFC Bank in the last 12 months is -1.56% on average per month, with a standard deviation of 6.64% and an annualized standard deviation of 23.01%. The minimum monthly return was -17.6% and the maximum monthly return was 6.95%.


'The return of HDFC Bank in the last 12 months is -1.56% on average per month, with a standard deviation of 6.64% and an annualized standard deviation of 23.01%. The minimum monthly return was -17.6% and the maximum monthly return was 6.95%.'

In [17]:
await ask('What is the return of TCS in comparison?')
end_session()


you> What is the return of TCS in comparison?
  · tool: lookup_ticker(query='TCS')
  · tool: fetch_stock_stats(months=12, ticker='TCS.NS')

alphascout> The return of TCS in the last 12 months is -3.49% on average per month, with a standard deviation of 6.93% and an annualized standard deviation of 24.02%. The minimum monthly return was -14.04% and the maximum monthly return was 5.87%. 

In comparison to HDFC Bank, TCS has a slightly lower average monthly return and a slightly higher annualized standard deviation, indicating a slightly higher risk. However, both stocks have similar maximum and minimum monthly returns.
[ended session: sess-1339cbe9]  (still persisted in sqlite+aiosqlite:///mfinagent.db)


## 9. Demo — Scenario 2: resume a prior session

Pick the most recent session id from `list_sessions()`, resume it, and
ask one follow-up. The agent should reuse the analyses from the prior
turns instead of re-fetching them.

In [18]:
sessions = await list_sessions()
prior_id = sessions[0].id  # most recent

    sess-1339cbe9  updated=1782746383.765918
    sess-ef2fd36b  updated=1782746315.25277


In [19]:
await resume_session(prior_id)
await ask('Of the stocks we have looked at so far, which has the lower risk?')

[resumed session: sess-1339cbe9]  (8 prior events)

you> Of the stocks we have looked at so far, which has the lower risk?

alphascout> HDFC Bank has a lower risk, with an annualized standard deviation of 23.01%, compared to TCS which has an annualized standard deviation of 24.02%.


'HDFC Bank has a lower risk, with an annualized standard deviation of 23.01%, compared to TCS which has an annualized standard deviation of 24.02%.'

In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('mfinagent.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)['name'].tolist()

for table in tables:
    print(f"\n{'='*20} Table: {table} {'='*20}")
    try:
        # Fetch a small sample to avoid truncation
        df = pd.read_sql_query(f"SELECT * FROM {table};", conn)
        if df.empty:
            print("[Empty Table]")
        else:
            # Print as a string to be more robust against display truncation
            print(df.to_string(index=False))
    except Exception as e:
        print(f"Error reading {table}: {e}")

conn.close()


==================== Table: adk_internal_metadata ====================
           key value
schema_version     1

==================== Table: sessions ====================
 app_name user_id            id state                create_time                update_time
mFinAgent    manu sess-ef2fd36b    {} 2026-06-29 15:18:21.144338 2026-06-29 15:18:35.252770
mFinAgent    manu sess-1339cbe9    {} 2026-06-29 15:19:30.328453 2026-06-29 15:20:17.580688

==================== Table: app_states ====================
 app_name state         update_time
mFinAgent    {} 2026-06-29 15:18:21

==================== Table: user_states ====================
 app_name user_id state         update_time
mFinAgent    manu    {} 2026-06-29 15:18:21

==================== Table: events ====================
                                  id  app_name user_id    session_id                          invocation_id                  timestamp                                                                             

In [21]:
import sqlite3
import json

# Connect and fetch all events from the events table
conn = sqlite3.connect('mfinagent.db')
cursor = conn.cursor()

# Updated query based on the actual table schema
cursor.execute("SELECT id, event_data FROM events")
rows = cursor.fetchall()

print(f"Found {len(rows)} events in the database.\n")

for i, (event_id, event_data) in enumerate(rows):
    print(f"--- Event {i+1} | ID: {event_id} ---")
    try:
        # Parse and pretty-print the JSON data
        parsed_data = json.loads(event_data)
        print(json.dumps(parsed_data, indent=2))
    except Exception as e:
        print(f"Raw Data (JSON parsing failed): {event_data}")
    print("\n" + "="*60 + "\n")

conn.close()

Found 18 events in the database.

--- Event 1 | ID: 9ec980da-718b-4f6d-948a-346e6f12e72f ---
{
  "content": {
    "parts": [
      {
        "text": "What is the return of HDFC Bank in the last 12 months?"
      }
    ],
    "role": "user"
  },
  "invocation_id": "e-fde331fa-47b0-45d8-94d2-cc2cb085edb4",
  "author": "user",
  "actions": {
    "state_delta": {},
    "artifact_delta": {},
    "requested_auth_configs": {},
    "requested_tool_confirmations": {}
  },
  "id": "9ec980da-718b-4f6d-948a-346e6f12e72f",
  "timestamp": 1782746301.161633
}


--- Event 2 | ID: 3d97abee-5e1e-476b-ab34-6ec535ec6c7a ---
{
  "model_version": "llama-3.3-70b-versatile",
  "content": {
    "parts": [
      {
        "function_call": {
          "id": "4mt8deevw",
          "args": {
            "query": "HDFC Bank"
          },
          "name": "lookup_ticker"
        }
      },
      {
        "function_call": {
          "id": "y2ez345be",
          "args": {
            "months": 12,
            "tick

## Recap

| Layer | Service | What's stored |
| --- | --- | --- |
| Conversation history + multi-turn replay | `DatabaseSessionService` (SQLite) | Full event list per session |

That's the whole storage story. ADK auto-replays prior events into the
LLM's context, so follow-up questions reuse earlier tool results without
re-fetching. No custom state keys, no memory service.

The same code, packaged as a runnable project (with an interactive CLI),
is in the `alphascout_adk/` folder next to this notebook.

*Educational. Not investment advice.*